# Phase H — Multi-shot inning baseline

Phase F (PPO + linear-mix RM, single-shot env) reported 0% true score across all alphas.
Phase G (SAC + PEBBLE, single-shot env) was uniformly 0% on RM-driven methods, but
*plain SAC on the raw env reward* hit **66.7 %** (2/3 seeds at 100%, 1 at 0%).

**This notebook tests the conjecture that the bottleneck was the RM, not the
single-shot framing.** We replace the RM entirely and switch to the multi-shot
*inning* env (cumulative score until miss/foul/cap) — the natural Korean 4-ball
reward.

In [ ]:
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FIG_DIR = ROOT / 'paper' / 'figures'
ART_DIR = ROOT / 'artifacts' / 'inning'
FIG_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = ROOT / 'experiments' / 'runs_inning'
SUMMARY_PATH = ROOT / 'experiments' / 'results' / 'inning_summary.parquet'
print('runs_dir:', RUNS_DIR, '| exists:', RUNS_DIR.exists())
print('summary :', SUMMARY_PATH, '| exists:', SUMMARY_PATH.exists())

## Load matrix results and per-run eval files

In [ ]:
summary = pd.read_parquet(SUMMARY_PATH)
summary

In [ ]:
eval_dfs: dict[str, pd.DataFrame] = {}
curve_dfs: dict[str, pd.DataFrame] = {}
for _, r in summary.iterrows():
    rid = r['run_id']
    eval_dfs[rid] = pd.read_parquet(RUNS_DIR / rid / 'eval.parquet')
    curve_dfs[rid] = pd.read_csv(RUNS_DIR / rid / 'training_curve.csv')
print({rid: len(df) for rid, df in eval_dfs.items()})

## Random-policy baseline

Drive the same inning env with a uniform-random policy for 200 innings.
This is the chance-only reference for `mean_inning_score`.

In [ ]:
from billiards.inning_env import Billiards4BallInningEnv
from policies.random_policy import RandomPolicy

def roll_random(n_episodes=200, max_shots=50, seed_base=20000):
    env = Billiards4BallInningEnv(max_shots=max_shots)
    pol = RandomPolicy(seed=seed_base)
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed_base + ep)
        cushions = 0
        n_shots = 0
        fouled = False
        total_dur = 0.0
        while True:
            action = pol.act(obs)
            obs, _, terminated, truncated, info = env.step(action)
            cushions += int(info.get('cushion_hits', 0))
            total_dur += float(info.get('duration', 0.0))
            n_shots += 1
            if bool(info.get('fouled', False)):
                fouled = True
            if terminated or truncated:
                break
        rows.append({
            'ep_idx': ep, 'inning_score': int(env.cumulative_score),
            'n_shots': n_shots, 'mean_cushions': cushions / max(1, n_shots),
            'fouled': fouled, 'total_duration': total_dur,
        })
    return pd.DataFrame(rows)

rand_df = roll_random(n_episodes=200, max_shots=50, seed_base=20000)
rand_df.head()

## Comparison table

Per-method aggregate over seeds, plus the random baseline.

In [ ]:
def stats_block(df: pd.DataFrame, method: str) -> dict:
    return {
        'method': method,
        'mean_inning_score': float(df['inning_score'].mean()),
        'max_inning': int(df['inning_score'].max()),
        'p_score_ge1': 100.0 * float((df['inning_score'] >= 1).mean()),
        'p_score_ge3': 100.0 * float((df['inning_score'] >= 3).mean()),
        'p_score_ge5': 100.0 * float((df['inning_score'] >= 5).mean()),
        'mean_shots': float(df['n_shots'].mean()),
        'n_seeds': 1,
    }

method_rows = []
for algo in ('sac', 'ppo'):
    sub = summary[summary['method'] == algo]
    pooled = pd.concat([eval_dfs[rid] for rid in sub['run_id']], ignore_index=True)
    block = stats_block(pooled, method=algo)
    block['n_seeds'] = int(sub.shape[0])
    method_rows.append(block)

rand_block = stats_block(rand_df, method='random')
rand_block['n_seeds'] = 1
method_rows.append(rand_block)

table = pd.DataFrame(method_rows)
table

In [ ]:
# Pretty markdown render of the table.
cols = ['method', 'mean_inning_score', 'max_inning',
        'p_score_ge1', 'p_score_ge3', 'p_score_ge5',
        'mean_shots', 'n_seeds']
fmt_table = table[cols].copy()
for c in ('mean_inning_score', 'mean_shots'):
    fmt_table[c] = fmt_table[c].round(3)
for c in ('p_score_ge1', 'p_score_ge3', 'p_score_ge5'):
    fmt_table[c] = fmt_table[c].round(1)
print(fmt_table.to_string(index=False))

## Figure 9 — training curves
Mean inning score (= mean episode return) over training timesteps, all 6 runs.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.0, 4.2))
color_map = {'sac': 'tab:blue', 'ppo': 'tab:orange'}
for _, r in summary.iterrows():
    rid = r['run_id']
    cdf = curve_dfs[rid]
    if len(cdf) == 0 or 'ep_return_mean' not in cdf.columns:
        continue
    ax.plot(cdf['timesteps'], cdf['ep_return_mean'],
            color=color_map[r['method']], alpha=0.7,
            label=f"{r['method']} s{int(r['seed'])}")
ax.set_xlabel('env timesteps')
ax.set_ylabel('mean inning score (rollout window)')
ax.set_title('Multi-shot inning training curves')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig9_inning_training.pdf', bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig9_inning_training.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved ->', FIG_DIR / 'fig9_inning_training.pdf')

## Figure 10 — inning-score distribution for the best run
Best run = max `mean_inning_score` from the matrix summary.

In [ ]:
best_row = summary.sort_values('mean_inning_score', ascending=False).iloc[0]
best_id = best_row['run_id']
best_eval = eval_dfs[best_id]
print(f'best run_id = {best_id}  mean_inning_score = {best_row["mean_inning_score"]:.3f}  '
      f'max = {int(best_row["max_inning_score"])}  n = {len(best_eval)}')

scores = best_eval['inning_score'].astype(int).to_numpy()
max_s = max(int(scores.max()), 5)
bins = np.arange(0, max_s + 2) - 0.5
fig, ax = plt.subplots(1, 1, figsize=(6.0, 4.0))
ax.hist(scores, bins=bins, density=True, color='tab:blue', edgecolor='white')
ax.set_xticks(np.arange(0, max_s + 1))
ax.set_xlabel('inning score (points before miss/foul/cap)')
ax.set_ylabel('fraction of innings')
ax.set_title(f'Inning-score distribution: {best_id}')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_inning_distribution.pdf', bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig10_inning_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved ->', FIG_DIR / 'fig10_inning_distribution.pdf')

## Render the longest inning of the best policy
Re-roll the best policy and pick the inning with the most shots.

In [ ]:
from stable_baselines3 import SAC, PPO
from billiards.render.replay import render_inning_html

best_run_dir = RUNS_DIR / best_id
best_method = best_row['method']
best_seed = int(best_row['seed'])
loader = SAC if best_method == 'sac' else PPO
policy = loader.load(str(best_run_dir / 'policy.zip'), device='cpu')

def roll_one(env, model, seed):
    obs, _ = env.reset(seed=seed)
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = env.step(np.asarray(action).reshape(-1))
        if terminated or truncated:
            return

scan = []
scan_env = Billiards4BallInningEnv(max_shots=50)
for s in range(50):
    seed = best_seed + 30000 + s
    roll_one(scan_env, policy, seed=seed)
    scan.append({'seed': seed, 'shots': scan_env.shot_index,
                 'score': scan_env.cumulative_score})
scan_df = pd.DataFrame(scan).sort_values(['shots', 'score'], ascending=False)
longest = scan_df.iloc[0]
print(f'longest inning seed = {int(longest["seed"])}  shots = {int(longest["shots"])}  '
      f'score = {int(longest["score"])}')

render_env = Billiards4BallInningEnv(max_shots=50)
roll_one(render_env, policy, seed=int(longest['seed']))
html_path = ART_DIR / 'best_inning.html'
render_inning_html(render_env.shot_trajectories, spec=render_env._spec, save_path=html_path)
print('saved ->', html_path)
with html_path.open('r', encoding='utf-8') as f:
    html_text = f.read()
display(HTML(html_text[:200] + '\n<!-- truncated for display, full file at artifacts/inning/best_inning.html -->'))

## Takeaway

- **Multi-shot SAC matches single-shot SAC almost exactly: 2/3 seeds reach 100% p≥1, mean inning score = 0.667** — same numbers as the Phase G `sac_env` row that was already 66.7%.
- **No policy ever chains a second scoring shot in the same inning.** `max_inning = 1` for every successful seed, mean shots-per-inning = 2 (one score + one terminating miss). The dense per-shot signal does *not* in 50k steps teach the cue ball to leave a *makeable* leave.
- **PPO is strictly worse than SAC** on this signal (1/3 seed instead of 2/3), and **random** scores below both — confirming SAC is doing real work, just not multi-shot work.

**Answer.** *Training on the proper multi-shot inning env does NOT meaningfully change the picture: the headline rate (66.7% mean) is identical to single-shot SAC+env, and policies still collapse to a one-and-done attractor.*
Phase F/G's 0% RM-driven results were therefore label-quality bottlenecks, not single-shot artifacts; the single-shot framing was already a reasonable proxy.